In [16]:
import pandas as pd
from pathlib import Path
import re
import time
import gc
import shutil

# ==================================================
# SETTINGS
# ==================================================

folder = Path("degree_year")

output_file = Path("IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv")

CHUNK_SIZE = 100_000

# ==================================================
# DELETE OLD OUTPUT FIRST
# Prevents old bad rows / NaN rows from staying
# ==================================================

if output_file.exists():
    output_file.unlink()
    print(f"Deleted old output file: {output_file}")

# ==================================================
# AWARD LEVEL MAP
# ==================================================

award_level_map = {
    1: "Certificate < 1 year",
    2: "Certificate 1-2 years",
    3: "Associate",
    4: "Certificate 2-4 years",
    5: "Bachelor",
    6: "Post-baccalaureate certificate",
    7: "Master",
    8: "Post-master certificate",
    9: "Doctor",
    10: "First-professional degree",
    11: "First-professional certificate",
    17: "Doctor - research/scholarship",
    18: "Doctor - professional practice",
    19: "Doctor - other",
}

def degree_group(x):
    if pd.isna(x):
        return "Unknown"

    try:
        x = int(x)
    except:
        return "Unknown"

    if x == 3:
        return "Associate"
    elif x == 5:
        return "Bachelor"
    elif x == 7:
        return "Master"
    elif x in [9, 17, 18, 19]:
        return "Doctor"
    elif x in [1, 2, 4, 6, 8, 10, 11]:
        return "Certificate/Other"
    else:
        return "Unknown"

# ==================================================
# HELPER FUNCTIONS
# ==================================================

def clean_text(value):
    """
    Cleans text and prevents NaN.
    """
    if pd.isna(value):
        return "Unknown"

    value = str(value).strip()

    if value.lower() in ["nan", "none", "null", ""]:
        return "Unknown"

    value = re.sub(r"\s+", " ", value).strip()

    if value == "":
        return "Unknown"

    return value


def find_column(df, possible_names):
    """
    Finds a column even if column names have different uppercase/lowercase.
    """
    col_map = {col.upper().strip(): col for col in df.columns}

    for name in possible_names:
        if name.upper().strip() in col_map:
            return col_map[name.upper().strip()]

    return None


def get_year_from_filename(filename):
    match = re.search(r"(19|20)\d{2}", filename)
    if match:
        return int(match.group())
    return pd.NA


def format_seconds(seconds):
    seconds = int(seconds)

    h = seconds // 3600
    m = (seconds % 3600) // 60
    s = seconds % 60

    if h > 0:
        return f"{h}h {m}m {s}s"
    elif m > 0:
        return f"{m}m {s}s"
    else:
        return f"{s}s"


def make_major_name(chunk, major_col, cip_col):
    """
    If major title exists, use it.
    If not, use CIP code.
    This prevents clean_major_name from becoming NaN.
    """
    if major_col:
        major = chunk[major_col].apply(clean_text)
    else:
        major = pd.Series(["Unknown"] * len(chunk), index=chunk.index)

    if cip_col:
        cip_clean = chunk[cip_col].apply(clean_text)
    else:
        cip_clean = pd.Series(["Unknown"] * len(chunk), index=chunk.index)

    # If major is Unknown but CIP exists, use CIP code as the major name
    major = major.where(
        major != "Unknown",
        "CIP " + cip_clean
    )

    # If CIP is also Unknown, keep Unknown
    major = major.replace("CIP Unknown", "Unknown")

    return major

# ==================================================
# FIND CSV FILES
# ==================================================

csv_files = sorted(folder.glob("*.csv"))

if not csv_files:
    raise FileNotFoundError(f"No CSV files found inside folder: {folder}")

print(f"Found {len(csv_files)} CSV files.")

# ==================================================
# CHECK DISK SPACE
# ==================================================

total, used, free = shutil.disk_usage(".")

print(f"Free disk space: {free / (1024**3):.2f} GB")
print("Starting clean combine...\n")

# ==================================================
# FIRST PASS: COLLECT ALL ORIGINAL COLUMNS
# This prevents mismatched columns between different years
# ==================================================

all_original_columns = []

for csv_file in csv_files:
    try:
        temp_header = pd.read_csv(
            csv_file,
            nrows=0,
            low_memory=False,
            encoding_errors="replace"
        )

        for col in temp_header.columns:
            if col not in all_original_columns:
                all_original_columns.append(col)

    except Exception as e:
        print(f"Could not read header from {csv_file.name}: {e}")

clean_columns = [
    "clean_year",
    "clean_award_level_code",
    "clean_award_level_name",
    "clean_degree_group",
    "clean_cip_code",
    "clean_major_name",
    "clean_degree_count"
]

output_columns = all_original_columns + clean_columns

print(f"Original columns found: {len(all_original_columns)}")
print(f"Final output columns: {len(output_columns)}\n")

# ==================================================
# PROCESS FILES ONE AT A TIME
# ==================================================

start_time = time.time()
total_rows_written = 0
header_written = False

for file_index, csv_file in enumerate(csv_files, start=1):
    file_start = time.time()
    year = get_year_from_filename(csv_file.name)

    print(f"Processing {file_index}/{len(csv_files)}: {csv_file.name}")

    try:
        reader = pd.read_csv(
            csv_file,
            chunksize=CHUNK_SIZE,
            low_memory=False,
            encoding_errors="replace"
        )

        file_rows = 0

        for chunk in reader:
            # -------------------------------
            # Find important columns
            # -------------------------------

            award_col = find_column(chunk, [
                "AWLEVEL",
                "AWARDLEVEL",
                "AWDLEVEL",
                "AWARD_LEVEL"
            ])

            cip_col = find_column(chunk, [
                "CIPCODE",
                "CIP_CODE",
                "CIP"
            ])

            major_col = find_column(chunk, [
                "CIPTITLE",
                "CIP_TITLE",
                "MAJOR_NAME",
                "PROGRAM",
                "PROGRAM_TITLE",
                "TITLE"
            ])

            count_col = find_column(chunk, [
                "CTOTALT",
                "TOTAL",
                "COUNT",
                "DEGREE_COUNT",
                "AWARDS",
                "COMPLETIONS"
            ])

            # -------------------------------
            # Add clean columns
            # -------------------------------

            chunk["clean_year"] = year

            if award_col:
                chunk["clean_award_level_code"] = pd.to_numeric(
                    chunk[award_col],
                    errors="coerce"
                )
                chunk["clean_award_level_name"] = (
                    chunk["clean_award_level_code"]
                    .map(award_level_map)
                    .fillna("Unknown")
                )
                chunk["clean_degree_group"] = chunk["clean_award_level_code"].apply(degree_group)
            else:
                chunk["clean_award_level_code"] = pd.NA
                chunk["clean_award_level_name"] = "Unknown"
                chunk["clean_degree_group"] = "Unknown"

            if cip_col:
                chunk["clean_cip_code"] = chunk[cip_col].apply(clean_text)
            else:
                chunk["clean_cip_code"] = "Unknown"

            chunk["clean_major_name"] = make_major_name(chunk, major_col, cip_col)

            if count_col:
                chunk["clean_degree_count"] = pd.to_numeric(
                    chunk[count_col],
                    errors="coerce"
                ).fillna(0).astype("int64")
            else:
                chunk["clean_degree_count"] = 0

            # -------------------------------
            # Make sure clean columns have no NaN
            # -------------------------------

            chunk["clean_year"] = chunk["clean_year"].fillna("Unknown")
            chunk["clean_award_level_name"] = chunk["clean_award_level_name"].fillna("Unknown")
            chunk["clean_degree_group"] = chunk["clean_degree_group"].fillna("Unknown")
            chunk["clean_cip_code"] = chunk["clean_cip_code"].fillna("Unknown")
            chunk["clean_major_name"] = chunk["clean_major_name"].fillna("Unknown")
            chunk["clean_degree_count"] = chunk["clean_degree_count"].fillna(0)

            # -------------------------------
            # Match final output column order
            # Missing old-year columns become blank
            # -------------------------------

            chunk = chunk.reindex(columns=output_columns)

            # -------------------------------
            # Write directly to output
            # -------------------------------

            chunk.to_csv(
                output_file,
                mode="a",
                index=False,
                header=not header_written
            )

            header_written = True

            file_rows += len(chunk)
            total_rows_written += len(chunk)

            # Clean memory after each chunk
            del chunk
            gc.collect()

        file_time = time.time() - file_start
        elapsed = time.time() - start_time

        avg_time_per_file = elapsed / file_index
        remaining_files = len(csv_files) - file_index
        eta = avg_time_per_file * remaining_files

        total, used, free = shutil.disk_usage(".")

        print(f"  Saved rows from file: {file_rows:,}")
        print(f"  File time: {format_seconds(file_time)}")
        print(f"  Total rows so far: {total_rows_written:,}")
        print(f"  Total time: {format_seconds(elapsed)}")
        print(f"  Estimated time left: {format_seconds(eta)}")
        print(f"  Free disk space now: {free / (1024**3):.2f} GB\n")

    except Exception as e:
        print(f"  ERROR with file: {csv_file.name}")
        print(f"  Reason: {e}\n")
        continue

# ==================================================
# DONE
# ==================================================

total_time = time.time() - start_time

print("DONE")
print(f"Output file created: {output_file}")
print(f"Total rows written: {total_rows_written:,}")
print(f"Total time: {format_seconds(total_time)}")

gc.collect()

Found 41 CSV files.
Free disk space: 532.72 GB
Starting clean combine...

Original columns found: 233
Final output columns: 240

Processing 1/41: 1984_c1984_cip.csv
  Saved rows from file: 114,727
  File time: 5s
  Total rows so far: 114,727
  Total time: 5s
  Estimated time left: 3m 39s
  Free disk space now: 532.68 GB

Processing 2/41: 1985_c1985_cip.csv
  Saved rows from file: 116,605
  File time: 5s
  Total rows so far: 231,332
  Total time: 11s
  Estimated time left: 3m 35s
  Free disk space now: 532.64 GB

Processing 3/41: 1986_c1986_cip.csv
  Saved rows from file: 117,479
  File time: 5s
  Total rows so far: 348,811
  Total time: 16s
  Estimated time left: 3m 32s
  Free disk space now: 532.61 GB

Processing 4/41: 1987_c1987_cip.csv
  Saved rows from file: 126,442
  File time: 6s
  Total rows so far: 475,253
  Total time: 23s
  Estimated time left: 3m 33s
  Free disk space now: 532.58 GB

Processing 5/41: 1988_c1988_cip.csv
  Saved rows from file: 131,154
  File time: 6s
  Total 

0

# do not use this due to overload memory problem
# print(df)
# print(final_dataset)
# df = pd.read_csv(file)

################################
# Use this instead:
# print(df.shape)
# print(df.head())
###################################
# data science project
# memory optimization 
# So yes, this is optimization, but it is memory optimization, not always speed optimization. It should prevent your computer from running out of space/memory.


In [19]:
import pandas as pd

preview = pd.read_csv(
    "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv",
    nrows=20,
    low_memory=False,
    keep_default_na=False
)

print(preview[[
    "clean_year",
    "clean_cip_code",
    "clean_major_name",
    "clean_award_level_name",
    "clean_degree_group",
    "clean_degree_count"
]])

    clean_year  clean_cip_code clean_major_name clean_award_level_name  \
0         1984           10102        CIP 10102               Bachelor   
1         1984           10102        CIP 10102                 Master   
2         1984           10103        CIP 10103               Bachelor   
3         1984           20201        CIP 20201               Bachelor   
4         1984           20301        CIP 20301               Bachelor   
5         1984           20399        CIP 20399                 Master   
6         1984           20501        CIP 20501               Bachelor   
7         1984           29999        CIP 29999               Bachelor   
8         1984           29999        CIP 29999                 Master   
9         1984           30499        CIP 30499               Bachelor   
10        1984           40301        CIP 40301               Bachelor   
11        1984           40301        CIP 40301                 Master   
12        1984           60101        

In [24]:
import pandas as pd
import time
from pathlib import Path

file = Path("IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv")

if not file.exists():
    raise FileNotFoundError("Output file not found. Check the file name/location.")

missing_count = 0
total_rows = 0
chunk_number = 0

start_time = time.time()

for chunk in pd.read_csv(
    file,
    usecols=["clean_major_name"],   # only read this column, faster
    chunksize=100_000,
    low_memory=False,
    keep_default_na=False
):
    chunk_number += 1
    total_rows += len(chunk)

    s = chunk["clean_major_name"].astype(str).str.strip().str.lower()

    missing_count += (
        (s == "") |
        (s.isin(["nan", "none", "null"]))
    ).sum()

    elapsed = time.time() - start_time

    print(
        f"Checked chunk {chunk_number} | rows checked: {total_rows:,} | bad rows: {missing_count:,} | time: {elapsed:.1f}s",
        flush=True
    )

print("====================================")
print("DONE")
print("Total rows checked:", total_rows)
print("Bad clean_major_name rows:", missing_count)

Checked chunk 1 | rows checked: 100,000 | bad rows: 0 | time: 0.2s
Checked chunk 2 | rows checked: 200,000 | bad rows: 0 | time: 0.4s
Checked chunk 3 | rows checked: 300,000 | bad rows: 0 | time: 0.6s
Checked chunk 4 | rows checked: 400,000 | bad rows: 0 | time: 0.8s
Checked chunk 5 | rows checked: 500,000 | bad rows: 0 | time: 1.0s
Checked chunk 6 | rows checked: 600,000 | bad rows: 0 | time: 1.2s
Checked chunk 7 | rows checked: 700,000 | bad rows: 0 | time: 1.4s
Checked chunk 8 | rows checked: 800,000 | bad rows: 0 | time: 1.6s
Checked chunk 9 | rows checked: 900,000 | bad rows: 0 | time: 1.8s
Checked chunk 10 | rows checked: 1,000,000 | bad rows: 0 | time: 2.0s
Checked chunk 11 | rows checked: 1,100,000 | bad rows: 0 | time: 2.2s
Checked chunk 12 | rows checked: 1,200,000 | bad rows: 0 | time: 2.4s
Checked chunk 13 | rows checked: 1,300,000 | bad rows: 0 | time: 2.6s
Checked chunk 14 | rows checked: 1,400,000 | bad rows: 0 | time: 2.8s
Checked chunk 15 | rows checked: 1,500,000 | ba

# REad all

In [27]:
import pandas as pd

file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

preview = pd.read_csv(
    file,
    nrows=20,
    low_memory=False,
    keep_default_na=False
)

display(preview)

,unitid,cipcode,awlevel,crace15,crace16,part,crace01,crace02,crace03,crace04,...,DVCWHW,CNRALW,CDISTEDP,clean_year,clean_award_level_code,clean_award_level_name,clean_degree_group,clean_cip_code,clean_major_name,clean_degree_count
0,100654,10102,5,11,0,,,,,,...,,,,1984,5,Bachelor,Bachelor,10102,CIP 10102,0
1,100654,10102,7,14,0,,,,,,...,,,,1984,7,Master,Master,10102,CIP 10102,0
2,100654,10103,5,1,1,,,,,,...,,,,1984,5,Bachelor,Bachelor,10103,CIP 10103,0
3,100654,20201,5,3,0,,,,,,...,,,,1984,5,Bachelor,Bachelor,20201,CIP 20201,0
4,100654,20301,5,3,0,,,,,,...,,,,1984,5,Bachelor,Bachelor,20301,CIP 20301,0
5,100654,20399,7,11,3,,,,,,...,,,,1984,7,Master,Master,20399,CIP 20399,0
6,100654,20501,5,2,0,,,,,,...,,,,1984,5,Bachelor,Bachelor,20501,CIP 20501,0
7,100654,29999,5,1,0,,,,,,...,,,,1984,5,Bachelor,Bachelor,29999,CIP 29999,0
8,100654,29999,7,10,0,,,,,,...,,,,1984,7,Master,Master,29999,CIP 29999,0
9,100654,30499,5,2,0,,,,,,...,,,,1984,5,Bachelor,Bachelor,30499,CIP 30499,0


In [29]:
import pandas as pd

file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

clean_cols = [
    "clean_year",
    "clean_cip_code",
    "clean_major_name",
    "clean_award_level_name",
    "clean_degree_group",
    "clean_degree_count"
]

preview = pd.read_csv(
    file,
    usecols=clean_cols,
    nrows=30,
    low_memory=False,
    keep_default_na=False
)

display(preview)

,clean_year,clean_award_level_name,clean_degree_group,clean_cip_code,clean_major_name,clean_degree_count
0,1984,Bachelor,Bachelor,10102,CIP 10102,0
1,1984,Master,Master,10102,CIP 10102,0
2,1984,Bachelor,Bachelor,10103,CIP 10103,0
3,1984,Bachelor,Bachelor,20201,CIP 20201,0
4,1984,Bachelor,Bachelor,20301,CIP 20301,0
5,1984,Master,Master,20399,CIP 20399,0
6,1984,Bachelor,Bachelor,20501,CIP 20501,0
7,1984,Bachelor,Bachelor,29999,CIP 29999,0
8,1984,Master,Master,29999,CIP 29999,0
9,1984,Bachelor,Bachelor,30499,CIP 30499,0


# Print only column names

In [31]:
import pandas as pd

file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

columns = pd.read_csv(file, nrows=0).columns.tolist()

for col in columns:
    print(col)

unitid
cipcode
awlevel
crace15
crace16
part
crace01
crace02
crace03
crace04
crace05
crace06
crace07
crace08
crace09
crace10
crace11
crace12
crace13
crace14
bal_m
bal_w
imprac01
imprac02
imprac03
imprac04
imprac05
imprac06
imprac07
imprac08
imprac09
imprac10
imprac11
imprac12
imprac13
imprac14
imprac15
imprac16
ix01
ix02
ix03
ix04
ix05
ix06
ix07
ix08
ix09
ix10
ix11
ix12
ix13
ix14
ix15
ix16
cbalm
cbalw
xcrace01
xcrace02
xcrace03
xcrace04
xcrace05
xcrace06
xcrace07
xcrace08
xcrace09
xcrace10
xcrace11
xcrace12
xcrace13
xcrace14
xcrace15
xcrace16
xcrace17
crace17
xcrace18
crace18
xcrace19
crace19
xcrace20
crace20
xcrace21
crace21
xcrace22
crace22
xcrace23
crace23
xcrace24
crace24
majornum
UNITID
CIPCODE
AWLEVEL
MAJORNUM
XCRACE01
CRACE01
XCRACE02
CRACE02
XCRACE03
CRACE03
XCRACE04
CRACE04
XCRACE05
CRACE05
XCRACE06
CRACE06
XCRACE07
CRACE07
XCRACE08
CRACE08
XCRACE09
CRACE09
XCRACE10
CRACE10
XCRACE11
CRACE11
XCRACE12
CRACE12
XCRACE13
CRACE13
XCRACE14
CRACE14
XCRACE15
CRACE15
XCRACE16
CRACE16
XCR

In [34]:
#4. View rows from a specific year only
import pandas as pd

file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

clean_cols = [
    "clean_year",
    "clean_cip_code",
    "clean_major_name",
    "clean_award_level_name",
    "clean_degree_group",
    "clean_degree_count"
]

rows_to_show = []

for chunk in pd.read_csv(
    file,
    usecols=clean_cols,
    chunksize=100_000,
    low_memory=False,
    keep_default_na=False
):
    chunk["clean_year"] = pd.to_numeric(chunk["clean_year"], errors="coerce")

    part = chunk[chunk["clean_year"] == 2022]

    if not part.empty:
        rows_to_show.append(part.head(20))

    if rows_to_show:
        break

result = pd.concat(rows_to_show, ignore_index=True)

display(result)

,clean_year,clean_award_level_name,clean_degree_group,clean_cip_code,clean_major_name,clean_degree_count
0,2022,Bachelor,Bachelor,1.0999,CIP 1.0999,9
1,2022,Bachelor,Bachelor,1.1001,CIP 1.1001,7
2,2022,Master,Master,1.1001,CIP 1.1001,7
3,2022,Doctor - research/scholarship,Doctor,1.1001,CIP 1.1001,3
4,2022,Bachelor,Bachelor,1.9999,CIP 1.9999,1
5,2022,Master,Master,1.9999,CIP 1.9999,8
6,2022,Doctor - research/scholarship,Doctor,1.9999,CIP 1.9999,3
7,2022,Bachelor,Bachelor,3.0599,CIP 3.0599,8
8,2022,Bachelor,Bachelor,4.0301,CIP 4.0301,3
9,2022,Master,Master,4.0301,CIP 4.0301,12


In [38]:
# View Bachelor rows only
import pandas as pd

file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

clean_cols = [
    "clean_year",
    "clean_cip_code",
    "clean_major_name",
    "clean_award_level_name",
    "clean_degree_group",
    "clean_degree_count"
]

rows_to_show = []

for chunk in pd.read_csv(
    file,
    usecols=clean_cols,
    chunksize=100_000,
    low_memory=False,
    keep_default_na=False
):
    part = chunk[chunk["clean_degree_group"] == "Bachelor"]

    if not part.empty:
        rows_to_show.append(part.head(30))

    if rows_to_show:
        break

result = pd.concat(rows_to_show, ignore_index=True)

display(result)

,clean_year,clean_award_level_name,clean_degree_group,clean_cip_code,clean_major_name,clean_degree_count
0,1984,Bachelor,Bachelor,10102,CIP 10102,0
1,1984,Bachelor,Bachelor,10103,CIP 10103,0
2,1984,Bachelor,Bachelor,20201,CIP 20201,0
3,1984,Bachelor,Bachelor,20301,CIP 20301,0
4,1984,Bachelor,Bachelor,20501,CIP 20501,0
5,1984,Bachelor,Bachelor,29999,CIP 29999,0
6,1984,Bachelor,Bachelor,30499,CIP 30499,0
7,1984,Bachelor,Bachelor,40301,CIP 40301,0
8,1984,Bachelor,Bachelor,60101,CIP 60101,0
9,1984,Bachelor,Bachelor,60201,CIP 60201,0


In [40]:
import pandas as pd

file = "IPEDS_ALL_ORIGINAL_INFO_CLEAN.csv"

clean_cols = [
    "clean_year",
    "clean_cip_code",
    "clean_major_name",
    "clean_award_level_name",
    "clean_degree_group",
    "clean_degree_count"
]

preview = pd.read_csv(
    file,
    usecols=clean_cols,
    nrows=50,
    low_memory=False,
    keep_default_na=False
)

display(preview)

,clean_year,clean_award_level_name,clean_degree_group,clean_cip_code,clean_major_name,clean_degree_count
0,1984,Bachelor,Bachelor,10102,CIP 10102,0
1,1984,Master,Master,10102,CIP 10102,0
2,1984,Bachelor,Bachelor,10103,CIP 10103,0
3,1984,Bachelor,Bachelor,20201,CIP 20201,0
4,1984,Bachelor,Bachelor,20301,CIP 20301,0
5,1984,Master,Master,20399,CIP 20399,0
6,1984,Bachelor,Bachelor,20501,CIP 20501,0
7,1984,Bachelor,Bachelor,29999,CIP 29999,0
8,1984,Master,Master,29999,CIP 29999,0
9,1984,Bachelor,Bachelor,30499,CIP 30499,0
